# ML-08 — Capstone Modeling Lane

This notebook trains a first honest model for the refresh lane. The goal is not complexity for its own sake; it is to beat the Week-4 baseline on the same data, the same split, and the same ranking metric.

## 1. Method choice and why

I am treating this as a binary classification problem with a ranking use case: predict `is_declining_label`, then sort by probability to decide which pages should be reviewed first. I start with Logistic Regression because it is readable, and I also train a Random Forest because this lane has non-linear interactions between freshness, visibility, and position that a tree ensemble can capture better than a straight line.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
 )
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

repo_root = next(
    (candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (candidate / "scripts" / "ml_utils.py").exists()),
    None,
 )
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

sys.path.insert(0, str(repo_root / "scripts"))
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, precision_at_k, write_json

feature_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
baseline_path = repo_root / "data" / "processed" / "baseline_refresh_queue.csv"

frame = pd.read_csv(feature_path)
baseline_frame = pd.read_csv(baseline_path)

RANDOM_STATE = 42

def build_feature_matrix(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    numeric_features = [column for column in MODEL_NUMERIC_FEATURES if column in frame.columns]
    categorical_features = [column for column in MODEL_CATEGORICAL_FEATURES if column in frame.columns]

    numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)

    categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dummy_na=False,
        dtype=float,
    )

    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    return feature_frame, list(feature_frame.columns)

def make_client_aware_split(
    frame: pd.DataFrame,
    target_series: pd.Series,
 ) -> tuple[np.ndarray, np.ndarray, str]:
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()

    if len(unique_clients) >= 5:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
        train_indices, test_indices = next(splitter.split(all_indices, target_series, groups=client_series))
        if (
            len(train_indices) > 0
            and len(test_indices) > 0
            and target_series.iloc[train_indices].nunique() == 2
            and target_series.iloc[test_indices].nunique() == 2
        ):
            return np.asarray(train_indices), np.asarray(test_indices), "client_holdout"

    train_indices, test_indices = train_test_split(
        all_indices,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=target_series,
    )
    return np.array(train_indices), np.array(test_indices), "stratified_row_holdout"

def build_models() -> dict[str, object]:
    return {
        "logistic_regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=1000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "decision_tree": DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=5,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        ),
        "random_forest": RandomForestClassifier(
            class_weight="balanced_subsample",
            max_depth=10,
            min_samples_leaf=25,
            n_estimators=200,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    }

def predict_probability(model: object, feature_frame: pd.DataFrame) -> np.ndarray:
    if not hasattr(model, "predict_proba"):
        raise TypeError(f"Model does not expose predict_proba: {type(model)!r}")
    probabilities = model.predict_proba(feature_frame)
    return np.asarray(probabilities[:, 1], dtype=float)

def metric_payload(
    target_series: pd.Series,
    probability_scores: np.ndarray,
    *,
    prefix: str = "",
 ) -> dict[str, float]:
    binary_predictions = (probability_scores >= 0.5).astype(int)
    payload = {
        f"{prefix}accuracy": float(accuracy_score(target_series, binary_predictions)),
        f"{prefix}precision": float(precision_score(target_series, binary_predictions, zero_division=0)),
        f"{prefix}recall": float(recall_score(target_series, binary_predictions, zero_division=0)),
        f"{prefix}f1": float(f1_score(target_series, binary_predictions, zero_division=0)),
        f"{prefix}precision_at_20": precision_at_k(target_series, probability_scores, 20),
        f"{prefix}precision_at_50": precision_at_k(target_series, probability_scores, 50),
        f"{prefix}precision_at_100": precision_at_k(target_series, probability_scores, 100),
    }
    if target_series.nunique() == 2:
        payload[f"{prefix}roc_auc"] = float(roc_auc_score(target_series, probability_scores))
        payload[f"{prefix}average_precision"] = float(average_precision_score(target_series, probability_scores))
    else:
        payload[f"{prefix}roc_auc"] = 0.0
        payload[f"{prefix}average_precision"] = 0.0
    return payload

def top_feature_importance(
    model: object,
    feature_columns: list[str],
    *,
    limit: int = 15,
 ) -> pd.DataFrame:
    if isinstance(model, Pipeline):
        classifier = model.named_steps["model"]
        raw_values = np.abs(classifier.coef_[0])
    elif hasattr(model, "feature_importances_"):
        raw_values = np.asarray(model.feature_importances_, dtype=float)
    else:
        raw_values = np.zeros(len(feature_columns), dtype=float)

    importance_frame = pd.DataFrame({"feature": feature_columns, "importance": raw_values})
    return importance_frame.sort_values("importance", ascending=False).head(limit).reset_index(drop=True)

feature_frame, feature_columns = build_feature_matrix(frame)
target_series = frame["is_declining_label"].astype(int)
train_indices, test_indices, split_strategy = make_client_aware_split(frame, target_series)

train_features = feature_frame.iloc[train_indices]
test_features = feature_frame.iloc[test_indices]
train_target = target_series.iloc[train_indices]
test_target = target_series.iloc[test_indices]

baseline_lookup = baseline_frame.set_index("content_id")["baseline_refresh_score"]
baseline_test_scores = frame.iloc[test_indices]["content_id"].map(baseline_lookup).fillna(0).to_numpy()

print(f"Rows: {len(frame):,}")
print(f"Target positive rate: {target_series.mean():.3f}")
print(f"Split strategy: {split_strategy}")
print(f"Train rows: {len(train_indices):,}")
print(f"Test rows: {len(test_indices):,}")
print(f"Feature columns: {len(feature_columns):,}")
display(frame[["content_id", "client_id", "is_declining_label"]].head(5))

Rows: 30,000
Target positive rate: 0.542
Split strategy: client_holdout
Train rows: 23,837
Test rows: 6,163
Feature columns: 52


,content_id,client_id,is_declining_label
0,content_304f48230142,client_f369cb89fc,1
1,content_a1fb4e703a9e,client_4e07408562,1
2,content_9aa793d4d895,client_7f2253d7e2,1
3,content_331d6c4de07b,client_19581e27de,0
4,content_d99b7a2d90ca,client_3fdba35f04,1


## 2. Split design

I use a client-aware holdout split so the same client does not appear in both train and test when I can avoid it. That is honest for this question because the model should generalize across content portfolios, not memorize one client's pattern. If there are too few clients for a clean grouped split, the notebook falls back to a stratified row split, but the grouped version is the default.

In [2]:
from sklearn.tree import DecisionTreeClassifier

def build_models() -> dict[str, object]:
    return {
        "logistic_regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=1000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "decision_tree": DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=5,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        ),
        "random_forest": RandomForestClassifier(
            class_weight="balanced_subsample",
            max_depth=10,
            min_samples_leaf=25,
            n_estimators=200,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    }

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
trained_models = build_models()
model_results: dict[str, dict[str, float]] = {}

for model_name, model in trained_models.items():
    model.fit(train_features, train_target)
    test_probabilities = predict_probability(model, test_features)
    model_results[model_name] = metric_payload(test_target, test_probabilities)

baseline_metrics = metric_payload(test_target, baseline_test_scores)

comparison_rows = [
    {
        "method": "baseline_rules",
        **baseline_metrics,
    },
]

for model_name, metrics in model_results.items():
    comparison_rows.append({"method": model_name, **metrics})

comparison_table = pd.DataFrame(comparison_rows)

best_model_name = sorted(
    model_results,
    key=lambda name: (
        model_results[name]["precision_at_50"],
        model_results[name]["average_precision"],
        model_results[name]["roc_auc"],
    ),
    reverse=True,
 )[0]

best_model = trained_models[best_model_name]
best_model_test_probabilities = predict_probability(best_model, test_features)
best_model_probabilities = predict_probability(best_model, feature_frame)

comparison_display = comparison_table[[
    "method",
    "roc_auc",
    "average_precision",
    "precision_at_20",
    "precision_at_50",
    "precision_at_100",
    "accuracy",
    "f1",
    "recall",
]]

display(comparison_display.sort_values("precision_at_50", ascending=False).reset_index(drop=True))
print(f"Best model by precision@50: {best_model_name}")

work_outputs = repo_root / "work" / "outputs"
work_outputs.mkdir(parents=True, exist_ok=True)

prediction_frame = frame[["content_id", "client_id", "is_declining_label"]].copy()
split_label = pd.Series("train", index=frame.index)
split_label.iloc[test_indices] = "test"
prediction_frame["split"] = split_label

for model_name, model in trained_models.items():
    prediction_frame[f"prob_{model_name}"] = predict_probability(model, feature_frame)

prediction_frame["best_model_name"] = best_model_name
prediction_frame["best_model_probability"] = best_model_probabilities

prediction_frame.to_csv(work_outputs / "model_predictions.csv", index=False)

results_payload = {
    "input_rows": int(len(frame)),
    "train_rows": int(len(train_indices)),
    "test_rows": int(len(test_indices)),
    "split_strategy": split_strategy,
    "target": "is_declining_label",
    "target_positive_rows": int(target_series.sum()),
    "target_positive_rate": float(target_series.mean()),
    "feature_count": int(len(feature_columns)),
    "model_numeric_features": [column for column in MODEL_NUMERIC_FEATURES if column in frame.columns],
    "model_categorical_features": [column for column in MODEL_CATEGORICAL_FEATURES if column in frame.columns],
    "baseline": baseline_metrics,
    "models": model_results,
    "best_model": {
        "name": best_model_name,
        "selection_metric": "precision_at_50",
        "feature_importance_top": top_feature_importance(best_model, feature_columns).to_dict(orient="records"),
    },
}

write_json(work_outputs / "model_results.json", results_payload)

display(top_feature_importance(best_model, feature_columns, limit=15))

,method,roc_auc,average_precision,precision_at_20,precision_at_50,precision_at_100,accuracy,f1,recall
0,logistic_regression,0.615521,0.604035,0.7,0.72,0.70,0.582509,0.605550,0.627183
1,decision_tree,0.612273,0.585004,0.4,0.54,0.65,0.586078,0.572482,0.542394
2,random_forest,0.609623,0.589574,0.5,0.54,0.60,0.582671,0.595343,0.600826
3,baseline_rules,0.497857,0.481863,0.4,0.32,0.31,0.467629,0.322247,0.247698


Best model by precision@50: logistic_regression


,feature,importance
0,log_impressions_90d,1.430558
1,log_clicks_90d,0.566999
2,word_count,0.514848
3,avg_position,0.385513
4,content_age_days,0.344896
5,log_sessions_90d,0.307488
6,char_count,0.302807
7,impression_tier_low,0.301505
8,word_count_tier_1000-2000,0.253652
9,position_tier_top_3,0.229653


## 4. Errors and interpretation

I trust the model only after I read the mistakes. The feature-importance table should tell me which signals it leans on, and the false positives / false negatives should show me where the ranking still gets confused. If the model wins by a little but makes ugly mistakes on the wrong kinds of pages, that matters more than the headline score.

In [4]:
test_prediction_frame = frame.iloc[test_indices][[
    "content_id",
    "client_id",
    "content_type",
    "is_declining_label",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]].copy()

test_prediction_frame["baseline_refresh_score"] = baseline_test_scores
test_prediction_frame["best_model_probability"] = best_model_test_probabilities
test_prediction_frame["predicted_label"] = (test_prediction_frame["best_model_probability"] >= 0.5).astype(int)
test_prediction_frame["error_type"] = np.select(
    [
        (test_prediction_frame["predicted_label"] == 1) & (test_prediction_frame["is_declining_label"] == 0),
        (test_prediction_frame["predicted_label"] == 0) & (test_prediction_frame["is_declining_label"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

error_rate_by_type = (
    test_prediction_frame.assign(error=(test_prediction_frame["predicted_label"] != test_prediction_frame["is_declining_label"]))
    .groupby("content_type", as_index=False)
    .agg(
        rows=("content_id", "size"),
        error_rate=("error", "mean"),
        avg_probability=("best_model_probability", "mean"),
        avg_baseline_score=("baseline_refresh_score", "mean"),
    )
    .sort_values("error_rate", ascending=False)
 )
display(error_rate_by_type)

false_positives = test_prediction_frame.loc[test_prediction_frame["error_type"] == "false_positive"].sort_values(
    "best_model_probability", ascending=False
 ).head(3)
false_negatives = test_prediction_frame.loc[test_prediction_frame["error_type"] == "false_negative"].sort_values(
    "best_model_probability", ascending=True
 ).head(3)

print("Three false positives: pages the model wants to review but the label says are not declining.")
display(false_positives)

print("Three false negatives: pages that are actually declining but the model is too conservative on.")
display(false_negatives)

top_feature_table = top_feature_importance(best_model, feature_columns, limit=10)
display(top_feature_table)

print(
    "The model is mostly leaning on visibility, freshness, and position-style signals, which is reasonable for a refresh lane. "
    "The hard errors are usually high-visibility pages that look risky but are not actually declining, or lower-volume pages where the decline signal is too weak for the model to separate cleanly. "
    "That is why the baseline comparison uses precision@50: the question is which pages rise to the top of the review queue, not just whether the model gets the average label right."
 )

,content_type,rows,error_rate,avg_probability,avg_baseline_score
0,keyword article,6163,0.417491,0.51256,0.392395


Three false positives: pages the model wants to review but the label says are not declining.


,content_id,client_id,content_type,is_declining_label,impressions_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update,baseline_refresh_score,best_model_probability,predicted_label,error_type
26614,content_7be5f150dc65,client_f369cb89fc,keyword article,0,290,2,0.0,5.9,96,20,0.342586,0.945758,1,false_positive
20736,content_41baf0722ad9,client_8527a891e2,keyword article,0,3115,4,0.0,12.8,275,104,0.705430,0.932267,1,false_positive
12869,content_5d5653c4eb4f,client_4e07408562,keyword article,0,15101,1,0.0,5.7,421,7,0.591982,0.917224,1,false_positive


Three false negatives: pages that are actually declining but the model is too conservative on.


,content_id,client_id,content_type,is_declining_label,impressions_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update,baseline_refresh_score,best_model_probability,predicted_label,error_type
8407,content_d1e915d03c28,client_4e07408562,keyword article,1,2,1,0.0,45.0,537,104,0.273842,0.050774,0,false_negative
29158,content_e18144cbd19d,client_4e07408562,keyword article,1,3,1,0.0,2.0,545,20,0.147410,0.055044,0,false_negative
17690,content_c268b1716236,client_e629fa6598,keyword article,1,3,2,0.0,41.7,502,20,0.133697,0.064743,0,false_negative


,feature,importance
0,log_impressions_90d,1.430558
1,log_clicks_90d,0.566999
2,word_count,0.514848
3,avg_position,0.385513
4,content_age_days,0.344896
5,log_sessions_90d,0.307488
6,char_count,0.302807
7,impression_tier_low,0.301505
8,word_count_tier_1000-2000,0.253652
9,position_tier_top_3,0.229653


The model is mostly leaning on visibility, freshness, and position-style signals, which is reasonable for a refresh lane. The hard errors are usually high-visibility pages that look risky but are not actually declining, or lower-volume pages where the decline signal is too weak for the model to separate cleanly. That is why the baseline comparison uses precision@50: the question is which pages rise to the top of the review queue, not just whether the model gets the average label right.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.